# 04 - SGD & 完整训练循环

## 目标
- 手写 SGD：
$$
W := W - \eta \cdot \frac{\partial L}{\partial W}
$$
- 自己实现 mini-batch：`for i in range(0, N, batch_size)`
- 组装完整训练循环：forward → loss → backward → update
- 打印 loss / accuracy 曲线，可视化第一层权重

## 网络结构
```
输入 (batch, 784)
  → Linear (784→128) + ReLU
  → Linear (128→10)
  → Softmax + CrossEntropyLoss
```

## 训练循环伪代码
```python
for epoch in range(epochs):
    for batch_x, batch_y in mini_batches:
        # forward
        z1 = linear1.forward(batch_x)
        a1 = relu.forward(z1)
        z2 = linear2.forward(a1)
        loss = softmax_cross_entropy(z2, batch_y)

        # backward
        dz2 = softmax_cross_entropy.backward()
        da1 = linear2.backward(dz2)
        dz1 = relu.backward(da1)
        linear1.backward(dz1)

        # update
        sgd.update(linear1)
        sgd.update(linear2)
```

In [1]:
%run 03_softmax_loss.ipynb

In [ ]:
# SGD（随机梯度下降） 优化器 + 准确率工具
"""
最原始的梯度下降，是要把所有训练数据都跑一遍再更新一次梯度，虽然准但是太慢
随机梯度下降（SGD）每次只用一个训练样本来更新一次梯度，虽然不太准但是快很多
最纯正的SDG是一次只用一个样本，但实际情况是每次用一个小批量（mini-batch），就是超参数batch_size
"""

class SGD:
    """最简单的随机梯度下降：W = W - lr * dW"""
    def __init__(self, lr=0.1):
        self.lr = lr

    def update(self, layer):
        layer.W -= self.lr * layer.dW
        layer.b -= self.lr * layer.db


def compute_accuracy(x, y):
    """
    跑一次 forward，算预测正确的比例
    为什么不用 loss_fn.forward() 而要手工算 softmax？
    因为 loss_fn.forward() 会覆盖 self.probs 和 self.y_true——如果训练循环里同时调了它，会搅乱 loss_fn的内部状态。这里只需要概率来做预测，没必要动 loss 状态，所以手工算最干净
    训练完的参数，都保存在这些class里面了
    """
    z1 = linear1.forward(x)
    a1 = relu.forward(z1)
    z2 = linear2.forward(a1)

    # 手工算softmax 取概率最大的类别
    z_stable = z2 - np.max(z2, axis=1, keepdims=True)
    probs = np.exp(z_stable) / np.sum(np.exp(z_stable), axis=1, keepdims=True)
    preds = np.argmax(probs, axis=1)    # 每个样本的预测类别
    true = np.argmax(y, axis=1)         # 把onehot标签转换成每个样本的真实类别
    return np.mean(preds == true)

In [3]:
# 组装模型 + 设置超参数
# 超参数
epochs = 10
batch_size = 64
lr = 0.1

# 组装网络
linear1 = Linear(784, 128)
relu = ReLU()
linear2 = Linear(128, 10)
loss_fn = SoftmaxCrossEntropy()
optimizer = SGD(lr)

print(f"参数数量: {linear1.W.size + linear1.b.size + linear2.W.size + linear2.b.size:,}")
# 784*128 + 128 + 128*10 + 10 = 100,362 + 1,290 = 101,652

参数数量: 101,770


In [ ]:
# 训练循环
train_losses = []
train_accs = []

for epoch in range(epochs):
    # 每 epoch 打乱数据顺序，相当于洗牌
    idx = np.random.permutation(len(x_train))
    x_shuffled = x_train[idx]       # 取第i个，组成新的数组，结果就是数据和标签对应的打乱
    y_shuffled = y_train[idx]

    epoch_loss = 0
    n_batches = 0

    for i in range(0, len(x_train), batch_size):        # (start, stop, step)不包含stop
        batch_x = x_shuffled[i:i + batch_size]          # 取64张
        batch_y = y_shuffled[i:i + batch_size]

        # forward
        z1 = linear1.forward(batch_x)
        a1 = relu.forward(z1)
        z2 = linear2.forward(a1)
        loss = loss_fn.forward(z2, batch_y)

        # backward 返回的一般是dx，继续向前传播的梯度
        dz2 = loss_fn.backward()
        da1 = linear2.backward(dz2)
        dz1 = relu.backward(da1)
        linear1.backward(dz1)       # 他不需要再往前传播了

        # update
        optimizer.update(linear1)
        optimizer.update(linear2)

        epoch_loss += loss          # 这个循环里的loss只是minibatch的，所以要累加
        n_batches += 1

    avg_loss = epoch_loss / n_batches
    train_losses.append(avg_loss)

    # 算准确率（用训练集采样，全算太慢）
    train_acc = compute_accuracy(x_train[:1000], y_train[:1000])    # 取前1000个，速度和准确性之间的平衡
    train_accs.append(train_acc)

    print(f"Epoch {epoch+1:2d}/{epochs} | loss: {avg_loss:.4f} | train acc: {train_acc:.4f}")

Epoch  1/10 | loss: 0.3699 | train acc: 0.9360
Epoch  2/10 | loss: 0.2016 | train acc: 0.9530
Epoch  3/10 | loss: 0.1522 | train acc: 0.9640
Epoch  4/10 | loss: 0.1234 | train acc: 0.9660
Epoch  5/10 | loss: 0.1041 | train acc: 0.9720
Epoch  6/10 | loss: 0.0895 | train acc: 0.9740
Epoch  7/10 | loss: 0.0787 | train acc: 0.9770
Epoch  8/10 | loss: 0.0696 | train acc: 0.9830
Epoch  9/10 | loss: 0.0624 | train acc: 0.9790
Epoch 10/10 | loss: 0.0566 | train acc: 0.9830


## 为什么准确率中间也会下降？

### 1. 只在 1000 张上测准确率

```python
train_acc = compute_accuracy(x_train[:1000], y_train[:1000])
```

1000 张不是完整的 60000 张。如果这一步的更新刚好让这 1000 张里的某几张预测从对变错，准确率就掉 0.4%，但其实全局准确率可能还在涨。就像抽查一样——有时候抽到的那几张刚好不顺。

### 2. SGD 的随机性

每个 batch 的梯度方向不完全一致：

- batch A（都是数字 3、7、8）→ 梯度偏向于「区分 3 和 8」
- batch B（都是数字 0、1、6）→ 梯度偏向于「区分 0 和 6」

上一个 batch 学到的让某个神经元关注水平笔画，下一个 batch 又把它推向别的方向。所以 loss 和 accuracy 都会抖动，不会是一条完美的单调曲线。

### 3. 学习率不变导致震荡

lr=0.1 在前期好使，在后期就太大了。训练没有 lr decay，所以 epochs 8–10 时 loss 已经很低了但还在震荡：接近最优解时步幅太大，跨过去又跨回来，反复横跳。

In [ ]:
# 画 loss 和 accuracy 曲线
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_losses, marker='o')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')

ax2.plot(train_accs, marker='o')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Training Accuracy')

plt.tight_layout()
plt.show()

In [6]:
# 测试集上评估最终模型性能
test_acc = compute_accuracy(x_test, y_test)
print(f"测试集准确率: {test_acc:.4f} ({test_acc*100:.2f}%)")

测试集准确率: 0.9751 (97.51%)


In [ ]:
# 取出 16 个神经元的权重，每个 reshape 回 28×28
fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for i, ax in enumerate(axes.flat):
    w = linear1.W[:, i].reshape(28, 28)
    ax.imshow(w, cmap='seismic', vmin=-0.5, vmax=0.5)
    ax.axis('off')
plt.suptitle('First Layer Weights (16 of 128 neurons)', fontsize=14)
plt.tight_layout()
plt.show()